In [2]:
import tkinter as tk
from tkinter import filedialog, ttk
import pygame
import os
from pathlib import Path

class MusicPlayer:
    def __init__(self, root):
        self.root = root
        self.root.title("Python Music Player")
        self.root.geometry("400x300")
        
        # Initialize pygame mixer
        pygame.mixer.init()
        
        # State variables
        self.current_song = None
        self.paused = False
        self.songs = []
        self.current_song_index = 0
        
        self.create_gui()
        
    def create_gui(self):
        # Create main frame
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Song information
        self.song_label = ttk.Label(main_frame, text="No song selected", wraplength=350)
        self.song_label.grid(row=0, column=0, columnspan=3, pady=10)
        
        # Buttons frame
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=1, column=0, columnspan=3, pady=10)
        
        # Control buttons
        ttk.Button(button_frame, text="Select Folder", command=self.select_folder).grid(row=0, column=0, padx=5)
        ttk.Button(button_frame, text="Play", command=self.play).grid(row=0, column=1, padx=5)
        ttk.Button(button_frame, text="Pause", command=self.pause).grid(row=0, column=2, padx=5)
        ttk.Button(button_frame, text="Stop", command=self.stop).grid(row=0, column=3, padx=5)
        ttk.Button(button_frame, text="Next", command=self.next_song).grid(row=0, column=4, padx=5)
        
        # Playlist frame
        playlist_frame = ttk.LabelFrame(main_frame, text="Playlist", padding="5")
        playlist_frame.grid(row=2, column=0, columnspan=3, sticky=(tk.W, tk.E, tk.N, tk.S), pady=10)
        
        # Playlist listbox
        self.playlist = tk.Listbox(playlist_frame, selectmode=tk.SINGLE, height=8)
        self.playlist.pack(fill=tk.BOTH, expand=True)
        
        # Volume control
        volume_frame = ttk.Frame(main_frame)
        volume_frame.grid(row=3, column=0, columnspan=3, pady=5)
        
        ttk.Label(volume_frame, text="Volume:").grid(row=0, column=0, padx=5)
        self.volume_scale = ttk.Scale(volume_frame, from_=0, to=1, orient=tk.HORIZONTAL,
                                    command=self.set_volume)
        self.volume_scale.set(0.5)  # Set default volume
        self.volume_scale.grid(row=0, column=1, padx=5)
        
    def select_folder(self):
        folder = filedialog.askdirectory()
        if folder:
            self.songs = []
            self.playlist.delete(0, tk.END)
            
            # Get all music files from the selected folder
            for file in Path(folder).glob("*.mp3"):  # You can add more formats here
                self.songs.append(str(file))
                self.playlist.insert(tk.END, file.name)
                
    def play(self):
        if not self.songs:
            return
            
        if self.paused:
            pygame.mixer.music.unpause()
            self.paused = False
        else:
            if not pygame.mixer.music.get_busy() or self.current_song != self.songs[self.current_song_index]:
                self.current_song = self.songs[self.current_song_index]
                pygame.mixer.music.load(self.current_song)
                pygame.mixer.music.play()
                self.song_label.config(text=f"Now playing: {Path(self.current_song).name}")
                
    def pause(self):
        if pygame.mixer.music.get_busy() and not self.paused:
            pygame.mixer.music.pause()
            self.paused = True
        
    def stop(self):
        pygame.mixer.music.stop()
        self.paused = False
        self.song_label.config(text="Playback stopped")
        
    def next_song(self):
        if not self.songs:
            return
            
        self.current_song_index = (self.current_song_index + 1) % len(self.songs)
        self.playlist.selection_clear(0, tk.END)
        self.playlist.selection_set(self.current_song_index)
        self.playlist.see(self.current_song_index)
        self.stop()
        self.play()
        
    def set_volume(self, value):
        pygame.mixer.music.set_volume(float(value))

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicPlayer(root)
    root.mainloop()